In [10]:
from FormUtils import pyForm, capture_physics_expr

In [11]:
%%pyForm Renormalization

* Process: Renormalization

#-
* Above suppresses extra output
Off Statistics;
Off FinalStats;
#include SquareAmplitude.h

Symbols e, Mmuon, Melec, x;
Vectors k, kf1, kf2;
Symbols k2 , kminusq2, kdotp1, kminusqdotp2, kdotkminusq;


Local MLO = (e^2) * (UB(i1, p3, Melec ) * g(i1, i2, mu1) * U(i2, p1, Melec))
            * phprop(mu1, mu2, q)
            * (UB(i3, p4, Mmuon) * g(i3, i4, mu2) * U(i4, p2, Mmuon));

Local MNLO = -1* (e^4) 
             * (UB(i1, p3, Melec) * g(i1, i2, mu1) * U(i2, p1, Melec)) 
             * phprop(mu1, mu3, q) 
             * g(i11, i12, mu3) * fprop(i12, i13, kf1, Melec) 
             * g(i13, i14, mu4) * fprop(i14, i11, kf2, Melec)
             * phprop(mu4, mu2, q)
             * (UB(i7, p4, Mmuon) * g(i7, i8, mu2) * U(i8, p2, Mmuon));

Local MTotal = MLO+MNLO;
#call squareamplitude(MLO, MsqLO)
#call squareamplitude(MNLO, MsqNLO)
#call squareamplitude(MTotal, MsqTotal)
.sort
Multiply 1/4;
.sort
Drop drop MsqNLO, MsqTotal, MLO, MNLO, MTotal ;
Local MInt = MsqTotal - MsqLO - MsqNLO;
.sort


* --- KINEMATIC DEFINITIONS: VACUUM POLARIZATION ---
* q  = p1 - p3           : Momentum transfer between electron and muon lines
* t  = q.q               : Mandelstam variable t (photon momentum squared)
* k  = loop momentum     : Integration variable for the fermion loop
* kf1 = k                : Momentum of the first fermion in the bubble
* kf2 = k - q            : Momentum of the second fermion in the bubble (cons. of momentum)

* Replace the propagator function with algebraic denominators
id prop(q.q) = 1/t;
id prop(-Melec^2 + kf1.kf1) = 1/(-Melec^2 +k2);
id prop(-Melec^2 + kf2.kf2) = 1/(-Melec^2 +kminusq2);

* Replace dot products involving loop momentum k
id kf1.p1? = kdotp1;
id kf2.p2? = kminusqdotp2;
id kf1.kf2 = kdotkminusq;
.sort

* --- MASSLESS APPROXIMATION ---
* keeps the Melec in the fermion propagator
id Melec = 0;
id Mmuon = 0;
#call Mandelstam2To2(p1,p2,p3,p4,0,0,0,0)
id u = -s -t;

* Save to file 
Format C;
#write <RenormalizationLO.txt> "%e;", MsqLO;
#write <Renormalization.txt> "%e;", MInt;
.sort
* Print 
Format;
Print+s MInt;
Print+s MsqLO;

.end

FORM 5.0.0 (Jan 27 2026, v5.0.0)                 Run: Fri Apr 17 01:18:35 2026
    
    * Process: Renormalization
    
    #-

   MsqLO =
       + 2*e^4
       + 4*s*t^(-1)*e^4
       + 4*s^2*t^(-2)*e^4
      ;

   MInt =
       + 192/(k2 - Melec^2)/(kminusq2 - Melec^2)*t^(-2)*e^6*kdotp1*
      kminusqdotp2
       + 128/(k2 - Melec^2)/(kminusq2 - Melec^2)*s*t^(-3)*e^6*kdotp1*
      kminusqdotp2
       - 32/(k2 - Melec^2)/(kminusq2 - Melec^2)*s*t^(-2)*e^6*kdotp1
       - 32/(k2 - Melec^2)/(kminusq2 - Melec^2)*s^2*t^(-3)*e^6*kdotp1
      ;




In [12]:
import numpy as np
import sympy as sp


form_expr_LO = capture_physics_expr("scripts/RenormalizationLO.txt")
s, t, u = sp.symbols("s t u") 
MLO = sp.expand(form_expr_LO)
print(f"MLO :")
sp.pprint(MLO)
print("\n")

form_expr_Int = capture_physics_expr("scripts/Renormalization.txt")
Melec = sp.symbols("Melec", real=True, positive=True)
t = sp.symbols("t", real=True) 
s = sp.symbols("s", real=True)
u = sp.symbols("u", real=True)
k2, kminusq2, kdotp1, kminusqdotp2 = sp.symbols(
    "k2 kminusq2 kdotp1 kminusqdotp2"
)

MInt_symbolic = sp.factor(form_expr_Int)
print(f"MInt :")
sp.pprint(MInt_symbolic,num_columns=100)
print("\n")


x = sp.symbols("x", real=True)
Delta = Melec**2 - x*(1 - x)*t
# I0 = Integral of 1/Delta^2
I0 = sp.integrate(1 / Delta**2, (x, 0, 1))
print(f"I0 :")
sp.pprint(I0,num_columns=140)
print("\n")

# I1 = Integral of x/Delta^2
I1 = sp.integrate(x / Delta**2, (x, 0, 1))
print(f"I1 :")
sp.pprint(I1,num_columns=140)
print("\n")

# I2 = Integral of x^2/Delta^2
I2 = sp.integrate(x**2 / Delta**2, (x, 0, 1))
print(f"I2 :")
sp.pprint(I2,num_columns=140)
print("\n")

#. log(Delta/mu^2) = log((Melec**2 - x(1-x)t) / mu^2)
#.  Taking the limit Melec -> 0: 
#    --> log(-t/mu^2 * x(1-x)) 
#    --> log(-t/mu^2) + log(x(1-x))
#  The log(-t/mu^2) term is a constant with respect to the x-integration
#  We avoid a bit complicated integrals by focusing on this term
#    
log_scale = sp.log(-t / mu2)
# Approximate L Master Integrals
L0 = log_scale * I0
L1 = log_scale * I1
L2 = log_scale * I2

print("L0:")
sp.pprint(L0)
print("\n")
print("L1:")
sp.pprint(L1)
print("\n")
print("L2:")
sp.pprint(L2)
print("\n")


MLO :
   4  2      4  2
2⋅e ⋅s    2⋅e ⋅u 
─────── + ───────
   2         2   
  t         t    


MInt :
    6        ⎛                                                           2    2    2⎞
16⋅e ⋅kdotp₁⋅⎝-4⋅kminusqdotp₂⋅s - 8⋅kminusqdotp₂⋅t + 4⋅kminusqdotp₂⋅u + s  - t  + u ⎠
─────────────────────────────────────────────────────────────────────────────────────
                        3 ⎛       2     ⎞ ⎛     2           ⎞                        
                       t ⋅⎝- Melec  + k₂⎠⋅⎝Melec  - kminusq₂⎠                        


I0 :
         -2          
─────────────────────
         4        2  
- 4⋅Melec  + Melec ⋅t


I1 :
                 2                        2          
          2⋅Melec                  2⋅Melec  - t      
- ──────────────────────── + ────────────────────────
           4          2  2            4          2  2
  - 4⋅Melec ⋅t + Melec ⋅t    - 4⋅Melec ⋅t + Melec ⋅t 


I2 :
                2                         2          
           Melec                